In [ ]:
!python3 -m venv .venv

In [ ]:
!.venv/bin/pip install ultralytics roboflow

In [ ]:
%pip install kaggle

In [ ]:
%pip install kagglehub

In [ ]:
%pip --list

In [ ]:
import kagglehub
cubeai_snake_detection_for_yolov8_path = kagglehub.dataset_download('cubeai/snake-detection-for-yolov8')

print('Data source import complete.')

In [ ]:
#!/bin/bash
!kaggle datasets download cubeai/snake-detection-for-yolov8

In [ ]:
import os

# Desativa o Weights & Biases diretamente no Python (evita erros do fish shell)
os.environ["WANDB_DISABLED"] = "true"
print("✅ Weights & Biases desativado com sucesso sem usar o shell!")


In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO
import yaml
import zipfile

# 1. Desativa o Weights & Biases de forma 100% segura com o terminal fish
os.environ["WANDB_DISABLED"] = "true"

# 2. Extrai o seu snake-detection-for-yolov8.zip local se ainda não foi descompactado
arquivo_zip = Path.cwd() / "snake-detection-for-yolov8.zip"
pasta_extracao = Path.cwd() / "dataset_cobras"

if not pasta_extracao.exists():
    print(f"📦 Extraindo {arquivo_zip.name} localmente...")
    with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:
        zip_ref.extractall(pasta_extracao)
    print("✅ Extração concluída!")

# 3. Prepara o arquivo YAML com os caminhos da sua máquina (substitui /kaggle/input/...)
data_yaml_orig = list(pasta_extracao.glob("**/data.yaml"))[0]
raiz_dataset = data_yaml_orig.parent

with open(data_yaml_orig, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["path"] = str(raiz_dataset)
config["train"] = "train/images"
config["val"] = "valid/images"
if "test" in config:
    config["test"] = "test/images"

# Muda o nome de '蛇' para 'Cobra' (evita os erros de UserWarning: Glyph 34503 na hora dos gráficos!)
config["names"] = ["Cobra"]

yaml_local = Path.cwd() / "dataset_pronto.yaml"
with open(yaml_local, "w", encoding="utf-8") as f:
    yaml.dump(config, f, allow_unicode=True)

print(
    f"🛠️ Configuração local salva! Total de imagens na validação: {len(list((raiz_dataset / 'valid' / 'images').glob('*.jpg')))} (Igual aos 1132 do Kaggle)"
)

# 4. Inicia o treinamento oficial com seed=42 e 100 épocas salvando na pasta local
pasta_resultados = Path.cwd() / "resultados_treino"
model = YOLO("yolov8n.pt")

print("\n🚀 INICIANDO TREINO DE 100 ÉPOCAS (SEED 42) NA SUA GPU...")
results = model.train(
    data=str(yaml_local),
    seed=42,  # Mesma semente matemática para buscar a reproducibilidade do Kaggle
    epochs=100,  # As 100 épocas necessárias
    project=str(pasta_resultados),  # Salva tudo dentro desta pasta local!
    name="yolov8_cobras",
    exist_ok=True,
)

# 5. Exibe a tabela comparativa ao término do treinamento
print("\n" + "=" * 70)
print("🎯 COMPARATIVO DO SEU TREINO LOCAL COM AS MÉTRICAS DO KAGGLE 🎯")
print("=" * 70)
print(f"Precisão (P):    {results.box.mp:.4f}   | Referência Kaggle: ~0.9050")
print(f"Revocação (R):   {results.box.mr:.4f}   | Referência Kaggle: ~0.8480")
print(f"mAP50 (B):       {results.box.map50:.4f}  | Referência Kaggle: ~0.9190")
print(f"mAP50-95 (B):    {results.box.map:.4f}    | Referência Kaggle: ~0.6150")
print("=" * 70)


In [ ]:
from ultralytics import YOLO

# Carrega o ponto de salvamento de onde parou na época 99
model = YOLO('resultados_treino/yolov8_cobras/weights/last.pt')

# Continua e finaliza a última época!
results = model.train(resume=True)


In [ ]:
print(cubeai_snake_detection_for_yolov8_path)

In [ ]:
print("O kernel está respondendo!")
